# Multi-agent Systems and Agentic Patterns with CrewAI

## Introduction

**CrewAI** is a powerful framework for orchestrating role-playing AI agents that work together to accomplish complex tasks. It provides a simple yet flexible API for creating multi-agent systems where each agent has specific roles, goals, and capabilities.

In this tutorial, we'll explore five fundamental agentic patterns using CrewAI with an **Azure OpenAI** deployment (`gpt-5.4-mini` or `gpt-5.6-luna`) as the model backend.

## 1. Setup & Installation

First, let's install the required libraries.

> **Note:** CrewAI supports Python 3.10–3.13 and pins many of its own dependency versions, so run this notebook in a fresh environment (for example a new Colab runtime or a dedicated virtual environment).

In [1]:
# Install required packages
# crewai: The main framework for multi-agent orchestration
# python-dotenv: Loads credentials from the .env file
# CrewAI supports Python 3.10-3.13

%pip install -q "crewai>=1.15" python-dotenv

# Windows only: pywin32 (used by the MCP SDK) normally needs a kernel restart after installation.
# Re-reading the site-packages .pth files makes it importable right away.
import site, sys, sysconfig
if sys.platform == "win32":
    for _site_dir in (sysconfig.get_paths()["purelib"], site.getusersitepackages()):
        site.addsitedir(_site_dir)

### Configure Azure OpenAI Credentials

This notebook uses an **Azure OpenAI** deployment through its OpenAI-compatible **v1 endpoint**.
Credentials are read from a `.env` file in this notebook's folder (or any parent folder). Copy `example.env` to `.env` and fill it in. On **Google Colab**, upload `.env` with the 📁 *Files* panel, or just enter the values when prompted:

| Variable | Description |
|----------|-------------|
| `AZURE_OPENAI_API_KEY` | Your Azure OpenAI API key |
| `AZURE_OPENAI_ENDPOINT` | Base URL, e.g. `https://<resource>.openai.azure.com/` (the full `.../openai/v1/` URL also works) |
| `AZURE_OPENAI_DEPLOYMENT` | Deployment (model) name: `gpt-5.4-mini` or `gpt-5.6-luna` |

If a value is missing, you'll be prompted for it.

**Security Best Practice**: Never hardcode API keys in your code. Keep them in `.env` (which must never be committed or shared).

In [2]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## 2. Model Configuration

Let's create a reusable LLM configuration using CrewAI's `LLM` class, pointed at our Azure OpenAI deployment. The `openai/` prefix selects CrewAI's native OpenAI provider, and `base_url` points it at the Azure OpenAI v1 endpoint.

In [3]:
from crewai import LLM

llm = LLM(
    model=f"openai/{AZURE_OPENAI_DEPLOYMENT}",  # Prefix with openai/ so it uses the OpenAI-compatible protocol
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

print(f"✅ Language model configured: {AZURE_OPENAI_DEPLOYMENT} (Azure OpenAI)")

✅ Language model configured: gpt-5.4-mini (Azure OpenAI)


## 3. Pattern A: Prompt Chaining

**Concept**: In prompt chaining, the output of one agent becomes the input for the next agent, creating a sequential pipeline of processing.

**Use Case**: When a task naturally breaks into sequential steps where each step depends on the previous one.

**Example**: Research → Summarization

> **Note:** Jupyter already runs an asyncio event loop, and CrewAI refuses to start a crew synchronously inside a running loop. In notebooks we therefore use `await crew.kickoff_async(...)`; in a plain Python script you can call `crew.kickoff(...)` instead.

In [4]:
from crewai import Agent, Task, Crew

# Agent A: Research Agent
# This agent gathers detailed information on a topic
research_agent = Agent(
    role="Research Specialist",
    goal="Gather comprehensive information about {topic}",
    backstory="You are an expert researcher with deep knowledge across multiple domains. "
              "You excel at finding relevant facts and insights.",
    llm=llm,
    verbose=True
)

# Agent B: Summary Agent
# This agent takes the research output and creates a concise summary
summary_agent = Agent(
    role="Content Summarizer",
    goal="Create a clear and concise summary of the research findings",
    backstory="You are a skilled writer who excels at distilling complex information "
              "into easily digestible summaries.",
    llm=llm,
    verbose=True
)

# Task 1: Research Task (Agent A)
# Output will be passed to the next task
research_task = Task(
    description="Research and gather detailed information about {topic}. "
                "Include key facts, recent developments, and important context.",
    agent=research_agent,
    expected_output="A comprehensive research document with key facts and insights"
)

# Task 2: Summary Task (Agent B)
# This task receives the output from research_task as context
summary_task = Task(
    description="Take the research findings and create a concise 3-paragraph summary. "
                "Focus on the most important points.",
    agent=summary_agent,
    expected_output="A clear 3-paragraph summary of the research findings",
    context=[research_task]  # This creates the chain: research_task output → summary_task input
)

# Create the crew with sequential process (default)
chain_crew = Crew(
    agents=[research_agent, summary_agent],
    tasks=[research_task, summary_task],
    verbose=True
)

# Execute the chain
print("=" * 60)
print("PATTERN A: PROMPT CHAINING EXAMPLE")
print("=" * 60)

result = await chain_crew.kickoff_async(inputs={"topic": "Quantum Computing"})
print("\n" + "=" * 60)
print("FINAL RESULT:")
print("=" * 60)
print(result)

PATTERN A: PROMPT CHAINING EXAMPLE


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.28                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ca752a7f-f7d2-4e96-bb5c-5f4f74c33916                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research and gather detailed information about Quantum Computing. Include key facts, recent              │
│  developments, and important context.                                                                           │
│  ID: bdc3db25-a482-42b6-8f94-cbf208811389                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research and gather detailed information about Quantum Computing. Include key facts, recent              │
│  developments, and important context.                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (5 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Summarizer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Quantum computing is a new computing paradigm that uses qubits, superposition, entanglement, and interference  │
│  to solve certain problems differently from classical computers. Its biggest promise is not general-purpose     │
│  speed for everyday tasks, but major advantages for specific areas such as cryptography, chemistry, materials   │
│  science, and some scientific computing problems. However, current systems are still in the NISQ era, meaning   │
│  they are noisy, error-prone, and limited by short coherence times, so the field’s long-term success depends    │
│  on building fault-tolerant quantum computers with reliable logical qubits.                                     │
│                                                                                                                 │
│  The research shows that the most important algorithms and impacts are Shor’s algorithm, which could break RSA  │
│  and elliptic-curve cryptography once large fault-tolerant machines exist, and Grover’s algorithm, which gives  │
│  a quadratic speedup for search. Quantum simulation is viewed as one of the most promising practical            │
│  applications because molecules and quantum systems are difficult for classical computers to model accurately.  │
│  At the same time, many other areas such as optimization and machine learning remain uncertain, with claims of  │
│  practical quantum advantage still unproven in most real-world settings.                                        │
│                                                                                                                 │
│  Hardware development is progressing across several platforms, including superconducting qubits, trapped ions,  │
│  neutral atoms, photonics, and silicon spin qubits, but scaling remains difficult because noise, decoherence,   │
│  and error-correction overhead are major barriers. The strongest near-term real-world implication is the        │
│  transition to post-quantum cryptography, since organizations must prepare now for future quantum attacks.      │
│  Overall, the research suggests that quantum computing is highly promising but still immature: progress is      │
│  real, yet broad commercial impact will likely come first in specialized scientific and security applications   │
│  before expanding more widely.                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Take the research findings and create a concise 3-paragraph summary. Focus on the most important         │
│  points.                                                                                                        │
│  Agent: Content Summarizer                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ca752a7f-f7d2-4e96-bb5c-5f4f74c33916                                                                       │
│  Final Output: Quantum computing is a new computing paradigm that uses qubits, superposition, entanglement,     │
│  and interference to solve certain problems differently from classical computers. Its biggest promise is not    │
│  general-purpose speed for everyday tasks, but major advantages for specific areas such as cryptography,        │
│  chemistry, materials science, and some scientific computing problems. However, current systems are still in    │
│  the NISQ era, meaning they are noisy, error-prone, and limited by short coherence times, so the field’s        │
│  long-term success depends on building fault-tolerant quantum computers with reliable logical qubits.           │
│                                                                                                                 │
│  The research shows that the most important algorithms and impacts are Shor’s algorithm, which could break RSA  │
│  and elliptic-curve cryptography once large fault-tolerant machines exist, and Grover’s algorithm, which gives  │
│  a quadratic speedup for search. Quantum simulation is viewed as one of the most promising practical            │
│  applications because molecules and quantum systems are difficult for classical computers to model accurately.  │
│  At the same time, many other areas such as optimization and machine learning remain uncertain, with claims of  │
│  practical quantum advantage still unproven in most real-world settings.                                        │
│                                                                                                                 │
│  Hardware development is progressing across several platforms, including superconducting qubits, trapped ions,  │
│  neutral atoms, photonics, and silicon spin qubits, but scaling remains difficult because noise, decoherence,   │
│  and error-correction overhead are major barriers. The strongest near-term real-world implication is the        │
│  transition to post-quantum cryptography, since organizations must prepare now for future quantum attacks.      │
│  Overall, the research suggests that quantum computing is highly promising but still immature: progress is      │
│  real, yet broad commercial impact will likely come first in specialized scientific and security applications   │
│  before expanding more widely.                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESULT:
Quantum computing is a new computing paradigm that uses qubits, superposition, entanglement, and interference to solve certain problems differently from classical computers. Its biggest promise is not general-purpose speed for everyday tasks, but major advantages for specific areas such as cryptography, chemistry, materials science, and some scientific computing problems. However, current systems are still in the NISQ era, meaning they are noisy, error-prone, and limited by short coherence times, so the field’s long-term success depends on building fault-tolerant quantum computers with reliable logical qubits.

The research shows that the most important algorithms and impacts are Shor’s algorithm, which could break RSA and elliptic-curve cryptography once large fault-tolerant machines exist, and Grover’s algorithm, which gives a quadratic speedup for search. Quantum simulation is viewed as one of the most promising practical applications because molecules and quantum sys

## 4. Pattern B: Routing

**Concept**: A router agent analyzes incoming requests and directs them to the appropriate specialized agent based on content or intent.

**Use Case**: When you have multiple specialized agents and need intelligent request distribution.

**Example**: Customer Service Router → Specialized Support Agents

In [5]:
# Specialized Agent 1: Technical Support
technical_agent = Agent(
    role="Technical Support Specialist",
    goal="Resolve technical issues and provide troubleshooting guidance",
    backstory="You are a technical expert with deep knowledge of software, hardware, "
              "and IT systems. You excel at diagnosing and solving technical problems.",
    llm=llm,
    verbose=True
)

# Specialized Agent 2: Sales Agent
sales_agent = Agent(
    role="Sales Representative",
    goal="Provide product information and help customers make purchase decisions",
    backstory="You are a knowledgeable sales professional who understands customer needs "
              "and can recommend the right products and services.",
    llm=llm,
    verbose=True
)

# Specialized Agent 3: General Information Agent
general_agent = Agent(
    role="General Information Specialist",
    goal="Answer general questions about company policies, hours, and basic information",
    backstory="You are a friendly customer service representative with comprehensive "
              "knowledge of company policies and general information.",
    llm=llm,
    verbose=True
)

# Router Agent: Analyzes queries and routes to appropriate specialist
router_agent = Agent(
    role="Customer Service Router",
    goal="Analyze customer queries and determine which specialist should handle them",
    backstory="You are an intelligent routing system that can quickly categorize customer "
              "inquiries as: TECHNICAL (tech issues), SALES (product/pricing questions), "
              "or GENERAL (policies/hours/basic info).",
    llm=llm,
    verbose=True
)

# Router Task: Determines which specialist to use
routing_task = Task(
    description="Analyze this customer query: '{query}'\n\n"
                "Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. "
                "Respond with ONLY one word: TECHNICAL, SALES, or GENERAL.",
    agent=router_agent,
    expected_output="One word: TECHNICAL, SALES, or GENERAL"
)

# Create tasks for each specialist
technical_task = Task(
    description="Handle this technical inquiry: '{query}'",
    agent=technical_agent,
    expected_output="Technical support response with solution steps"
)

sales_task = Task(
    description="Handle this sales inquiry: '{query}'",
    agent=sales_agent,
    expected_output="Sales response with product recommendations"
)

general_task = Task(
    description="Handle this general inquiry: '{query}'",
    agent=general_agent,
    expected_output="General information response"
)

# Function to route queries
async def route_query(query):
    """Route a query to the appropriate specialist agent"""
    print("=" * 60)
    print(f"ROUTING QUERY: {query}")
    print("=" * 60)

    # First, determine the route
    route_crew = Crew(
        agents=[router_agent],
        tasks=[routing_task],
        verbose=True
    )

    route_result = await route_crew.kickoff_async(inputs={"query": query})
    route = str(route_result).strip().upper()

    print(f"\n→ Route determined: {route}\n")

    # Then, execute the appropriate specialist task
    if "TECHNICAL" in route:
        specialist_crew = Crew(agents=[technical_agent], tasks=[technical_task], verbose=True)
    elif "SALES" in route:
        specialist_crew = Crew(agents=[sales_agent], tasks=[sales_task], verbose=True)
    else:
        specialist_crew = Crew(agents=[general_agent], tasks=[general_task], verbose=True)

    result = await specialist_crew.kickoff_async(inputs={"query": query})

    print("\n" + "=" * 60)
    print("FINAL RESPONSE:")
    print("=" * 60)
    print(result)
    return result

# Test the routing pattern
print("\n\n" + "=" * 60)
print("PATTERN B: ROUTING EXAMPLE")
print("=" * 60 + "\n")

# Test with a technical query
await route_query("My laptop won't connect to WiFi. How do I fix this?")



PATTERN B: ROUTING EXAMPLE

ROUTING QUERY: My laptop won't connect to WiFi. How do I fix this?


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.28                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9e060f06-771e-4de8-b418-a8c4cbbf9cb6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze this customer query: 'My laptop won't connect to WiFi. How do I fix this?'                       │
│                                                                                                                 │
│  Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. Respond with ONLY one word: TECHNICAL, SALES, or  │
│  GENERAL.                                                                                                       │
│  ID: 3b9f31a1-1c41-4962-94d3-59c7c4927a0e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Service Router                                                                                 │
│                                                                                                                 │
│  Task: Analyze this customer query: 'My laptop won't connect to WiFi. How do I fix this?'                       │
│                                                                                                                 │
│  Determine if this is a TECHNICAL, SALES, or GENERAL inquiry. Respond with ONLY one word: TECHNICAL, SALES, or  │
│  GENERAL.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (10 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Handle this technical inquiry: 'My laptop won't connect to WiFi. How do I fix this?'                     │
│  Agent: Technical Support Specialist                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESPONSE:
Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi. Start from the top and stop when it works.

## 1) Check the basics
- Make sure **Wi‑Fi is turned on** on the laptop.
- Confirm **Airplane mode is off**.
- Restart your **laptop** and **Wi‑Fi router/modem**.
  - Unplug the router/modem for **30 seconds**, then plug it back in.
- Check whether **other devices** can connect to the same Wi‑Fi network.
  - If no devices can connect, the problem is likely the router or internet service.

## 2) Move closer to the router
- Try connecting within a few feet of the router.
- Weak signal or interference can prevent a connection.

## 3) Forget and reconnect to the network
### Windows
1. Open **Settings** → **Network & Internet** → **Wi‑Fi**.
2. Select **Manage known networks**.
3. Choose your network and click **Forget**.
4. Reconnect and re-enter the Wi‑Fi password.

### Mac
1. Go to **System Settings** → **Wi‑Fi**.
2. Find the network and remove/forget

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a93184e7-da08-41bd-abab-e26abd6300d0                                                                       │
│  Final Output: Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi. Start from the    │
│  top and stop when it works.                                                                                    │
│                                                                                                                 │
│  ## 1) Check the basics                                                                                         │
│  - Make sure **Wi‑Fi is turned on** on the laptop.                                                              │
│  - Confirm **Airplane mode is off**.                                                                            │
│  - Restart your **laptop** and **Wi‑Fi router/modem**.                                                          │
│    - Unplug the router/modem for **30 seconds**, then plug it back in.                                          │
│  - Check whether **other devices** can connect to the same Wi‑Fi network.                                       │
│    - If no devices can connect, the problem is likely the router or internet service.                           │
│                                                                                                                 │
│  ## 2) Move closer to the router                                                                                │
│  - Try connecting within a few feet of the router.                                                              │
│  - Weak signal or interference can prevent a connection.                                                        │
│                                                                                                                 │
│  ## 3) Forget and reconnect to the network                                                                      │
│  ### Windows                                                                                                    │
│  1. Open **Settings** → **Network & Internet** → **Wi‑Fi**.                                                     │
│  2. Select **Manage known networks**.                                                                           │
│  3. Choose your network and click **Forget**.                                                                   │
│  4. Reconnect and re-enter the Wi‑Fi password.                                                                  │
│                                                                                                                 │
│  ### Mac                                                                                                        │
│  1. Go to **System Settings** → **Wi‑Fi**.                                                                      │
│  2. Find the network and remove/forget it if available.                                                         │
│  3. Rejoin the network and enter the password again.                                                            │
│                                                                                                                 │
│  ## 4) Check the password and security settings                                                                 │
│  - Make sure you’re using the **correct Wi‑Fi password**.                                                       │
│  - If the router was recently changed, the password ma

CrewOutput(raw='Here are the most effective steps to fix a laptop that won’t connect to Wi‑Fi. Start from the top and stop when it works.\n\n## 1) Check the basics\n- Make sure **Wi‑Fi is turned on** on the laptop.\n- Confirm **Airplane mode is off**.\n- Restart your **laptop** and **Wi‑Fi router/modem**.\n  - Unplug the router/modem for **30 seconds**, then plug it back in.\n- Check whether **other devices** can connect to the same Wi‑Fi network.\n  - If no devices can connect, the problem is likely the router or internet service.\n\n## 2) Move closer to the router\n- Try connecting within a few feet of the router.\n- Weak signal or interference can prevent a connection.\n\n## 3) Forget and reconnect to the network\n### Windows\n1. Open **Settings** → **Network & Internet** → **Wi‑Fi**.\n2. Select **Manage known networks**.\n3. Choose your network and click **Forget**.\n4. Reconnect and re-enter the Wi‑Fi password.\n\n### Mac\n1. Go to **System Settings** → **Wi‑Fi**.\n2. Find the net

## 5. Pattern C: Parallelization

**Concept**: Multiple agents work simultaneously on different aspects of a problem, then their results are aggregated.

**Use Case**: When you have independent sub-tasks that can be executed concurrently to save time.

**Example**: Market Research + Competitor Analysis → Unified Report

**How**: Tasks marked with `async_execution=True` run concurrently. A later task that lists them in `context` waits for all of them to finish before it starts.

In [6]:
from crewai import Process

# Agent 1: Market Research Agent (runs in parallel)
market_researcher = Agent(
    role="Market Research Analyst",
    goal="Analyze market trends, customer needs, and opportunities for {product}",
    backstory="You are a market research expert with years of experience in analyzing "
              "market dynamics, consumer behavior, and industry trends.",
    llm=llm,
    verbose=True
)

# Agent 2: Competitor Analysis Agent (runs in parallel)
competitor_analyst = Agent(
    role="Competitive Intelligence Analyst",
    goal="Analyze competitors, their strategies, and market positioning for {product}",
    backstory="You are a competitive intelligence specialist who excels at understanding "
              "competitor strengths, weaknesses, and market strategies.",
    llm=llm,
    verbose=True
)

# Agent 3: Aggregator Agent (runs after parallel tasks complete)
aggregator = Agent(
    role="Strategic Report Writer",
    goal="Combine market research and competitor analysis into a unified strategic report",
    backstory="You are a strategic analyst who excels at synthesizing multiple sources "
              "of information into actionable insights.",
    llm=llm,
    verbose=True
)

# Parallel Task 1: Market Research
market_task = Task(
    description="Conduct market research for {product}. Analyze:\n"
                "1. Market size and growth potential\n"
                "2. Target customer segments\n"
                "3. Key market trends\n"
                "4. Opportunities and challenges",
    agent=market_researcher,
    expected_output="Comprehensive market research report with key insights",
    async_execution=True,  # Runs concurrently with competitor_task
)

# Parallel Task 2: Competitor Analysis
competitor_task = Task(
    description="Analyze competitors in the {product} market. Include:\n"
                "1. Main competitors\n"
                "2. Their strengths and weaknesses\n"
                "3. Market share and positioning\n"
                "4. Competitive advantages",
    agent=competitor_analyst,
    expected_output="Detailed competitor analysis report",
    async_execution=True,  # Runs concurrently with market_task
)

# Aggregation Task: Combines results from both parallel tasks
aggregation_task = Task(
    description="Synthesize the market research and competitor analysis into a unified "
                "strategic report. Provide actionable recommendations.",
    agent=aggregator,
    expected_output="Unified strategic report with recommendations",
    context=[market_task, competitor_task]  # Waits for both tasks to complete
)

# Create the crew: both async tasks start at the same time, and the
# aggregation task waits for their results (via `context`) before it runs
parallel_crew = Crew(
    agents=[market_researcher, competitor_analyst, aggregator],
    tasks=[market_task, competitor_task, aggregation_task],
    verbose=True,
    process=Process.sequential  # Tasks 1 & 2 run concurrently; Task 3 aggregates
)

# Execute parallel pattern
print("=" * 60)
print("PATTERN C: PARALLELIZATION EXAMPLE")
print("=" * 60)
print("Note: Market research and competitor analysis run concurrently (async_execution=True)")
print("=" * 60 + "\n")

result = await parallel_crew.kickoff_async(inputs={"product": "AI-powered fitness app"})

print("\n" + "=" * 60)
print("AGGREGATED RESULT:")
print("=" * 60)
print(result)

PATTERN C: PARALLELIZATION EXAMPLE
Note: Market research and competitor analysis run concurrently (async_execution=True)



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.28                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a502356e-ad2b-455c-92e5-70bfdec7ff34                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Conduct market research for AI-powered fitness app. Analyze:                                             │
│  1. Market size and growth potential                                                                            │
│  2. Target customer segments                                                                                    │
│  3. Key market trends                                                                                           │
│  4. Opportunities and challenges                                                                                │
│  ID: b6458762-0579-4bcd-a427-e62d7bc27701                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze competitors in the AI-powered fitness app market. Include:                                       │
│  1. Main competitors                                                                                            │
│  2. Their strengths and weaknesses                                                                              │
│  3. Market share and positioning                                                                                │
│  4. Competitive advantages                                                                                      │
│  ID: 3b724fd3-5387-4e3d-bc21-9b174899f929                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Conduct market research for AI-powered fitness app. Analyze:                                             │
│  1. Market size and growth potential                                                                            │
│  2. Target customer segments                                                                                    │
│  3. Key market trends                                                                                           │
│  4. Opportunities and challenges                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (10 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Strategic Report Writer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Unified Strategic Report: AI-Powered Fitness App Market                                                      │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  The AI-powered fitness app market is attractive, fast-growing, and crowded. Demand is being driven by          │
│  consumer appetite for personalized coaching, convenience, low-cost fitness solutions, and integrated wellness  │
│  experiences. AI meaningfully improves the value proposition by enabling adaptive programming, form feedback,   │
│  recovery-aware recommendations, nutrition support, and ongoing motivation at scale.                            │
│                                                                                                                 │
│  However, this is not a market where generic execution wins. The field is fragmented across AI-first coaching   │
│  apps, premium connected fitness ecosystems, wearable-driven performance platforms, and broad wellness brands.  │
│  The biggest commercial risks are weak differentiation, high acquisition costs, and rapid churn after initial   │
│  novelty fades.                                                                                                 │
│                                                                                                                 │
│  The strategic conclusion is clear:                                                                             │
│  **Do not build a broad “AI fitness app” for everyone. Build a focused outcome-based product for a specific     │
│  segment, then use AI to deliver better personalization, adherence, and results than existing competitors.**    │
│                                                                                                                 │
│  The best near-term opportunity is to target a segment with:                                                    │
│  - clear pain points,                                                                                           │
│  - strong willingness to pay,                                                                                   │
│  - measurable outcomes,                                                                                         │
│  - and enough repeat usage to support subscription retention.                                                   │
│                                                                                                                 │
│  The strongest segment opportunities are:                                                                       │
│  1. beginners / restart users,                                                                                  │
│  2. busy professionals,                                                                                         │
│  3. weight-loss and transformation users,                                                                       │
│  4. women’s fitness,                                                                                            │
│  5. strength training users,                           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Synthesize the market research and competitor analysis into a unified strategic report. Provide          │
│  actionable recommendations.                                                                                    │
│  Agent: Strategic Report Writer                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a502356e-ad2b-455c-92e5-70bfdec7ff34                                                                       │
│  Final Output: # Unified Strategic Report: AI-Powered Fitness App Market                                        │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  The AI-powered fitness app market is attractive, fast-growing, and crowded. Demand is being driven by          │
│  consumer appetite for personalized coaching, convenience, low-cost fitness solutions, and integrated wellness  │
│  experiences. AI meaningfully improves the value proposition by enabling adaptive programming, form feedback,   │
│  recovery-aware recommendations, nutrition support, and ongoing motivation at scale.                            │
│                                                                                                                 │
│  However, this is not a market where generic execution wins. The field is fragmented across AI-first coaching   │
│  apps, premium connected fitness ecosystems, wearable-driven performance platforms, and broad wellness brands.  │
│  The biggest commercial risks are weak differentiation, high acquisition costs, and rapid churn after initial   │
│  novelty fades.                                                                                                 │
│                                                                                                                 │
│  The strategic conclusion is clear:                                                                             │
│  **Do not build a broad “AI fitness app” for everyone. Build a focused outcome-based product for a specific     │
│  segment, then use AI to deliver better personalization, adherence, and results than existing competitors.**    │
│                                                                                                                 │
│  The best near-term opportunity is to target a segment with:                                                    │
│  - clear pain points,                                                                                           │
│  - strong willingness to pay,                                                                                   │
│  - measurable outcomes,                                                                                         │
│  - and enough repeat usage to support subscription retention.                                                   │
│                                                                                                                 │
│  The strongest segment opportunities are:                                                                       │
│  1. beginners / restart users,                                                                                  │
│  2. busy professionals,                                                                                         │
│  3. weight-loss and transformation users,                                                                       │
│  4. women’s fitness,                                                                                            │
│  5. strength training users,                          


AGGREGATED RESULT:
# Unified Strategic Report: AI-Powered Fitness App Market

## Executive Summary

The AI-powered fitness app market is attractive, fast-growing, and crowded. Demand is being driven by consumer appetite for personalized coaching, convenience, low-cost fitness solutions, and integrated wellness experiences. AI meaningfully improves the value proposition by enabling adaptive programming, form feedback, recovery-aware recommendations, nutrition support, and ongoing motivation at scale.

However, this is not a market where generic execution wins. The field is fragmented across AI-first coaching apps, premium connected fitness ecosystems, wearable-driven performance platforms, and broad wellness brands. The biggest commercial risks are weak differentiation, high acquisition costs, and rapid churn after initial novelty fades.

The strategic conclusion is clear:  
**Do not build a broad “AI fitness app” for everyone. Build a focused outcome-based product for a specific segme

## 6. Pattern D: Orchestrator-Worker

**Concept**: A manager/orchestrator agent breaks down complex tasks into subtasks and delegates them to worker agents sequentially.

**Use Case**: When a complex task needs to be decomposed into smaller, manageable steps that must be executed in order.

**Example**: Manager Agent → Worker Agent (Research → Outline → Draft → Edit)

In [7]:
# Manager Agent: Orchestrates the overall process
manager_agent = Agent(
    role="Project Manager",
    goal="Break down the task of writing a blog post about {topic} into clear subtasks",
    backstory="You are an experienced project manager who excels at decomposing complex "
              "tasks into manageable steps and ensuring quality execution.",
    llm=llm,
    verbose=True
)

# Worker Agent: Executes the subtasks
worker_agent = Agent(
    role="Content Worker",
    goal="Execute assigned subtasks efficiently and effectively",
    backstory="You are a versatile content professional capable of research, writing, "
              "and editing. You follow instructions precisely and deliver quality work.",
    llm=llm,
    verbose=True
)

# Step 1: Manager plans the work
planning_task = Task(
    description="Create a plan for writing a blog post about {topic}. "
                "List the key steps needed (research, outline, draft, edit).",
    agent=manager_agent,
    expected_output="A clear plan with 4 steps for creating the blog post"
)

# Step 2: Worker conducts research
research_step = Task(
    description="SUBTASK 1: Research {topic}. Gather key facts, statistics, and insights "
                "that would be valuable for a blog post.",
    agent=worker_agent,
    expected_output="Research findings with key facts and insights",
    context=[planning_task]
)

# Step 3: Worker creates outline
outline_step = Task(
    description="SUBTASK 2: Based on the research, create a detailed outline for a blog post "
                "about {topic}. Include main sections and key points.",
    agent=worker_agent,
    expected_output="Detailed blog post outline with sections and key points",
    context=[research_step]
)

# Step 4: Worker writes draft
draft_step = Task(
    description="SUBTASK 3: Using the outline, write a complete draft of the blog post. "
                "Make it engaging and informative (approximately 300 words).",
    agent=worker_agent,
    expected_output="Complete blog post draft",
    context=[outline_step]
)

# Step 5: Worker edits and finalizes
edit_step = Task(
    description="SUBTASK 4: Review and edit the draft. Improve clarity, fix any issues, "
                "and ensure the blog post is polished and ready for publication.",
    agent=worker_agent,
    expected_output="Final, polished blog post ready for publication",
    context=[draft_step]
)

# Create the orchestrator-worker crew
orchestrator_crew = Crew(
    agents=[manager_agent, worker_agent],
    tasks=[planning_task, research_step, outline_step, draft_step, edit_step],
    verbose=True,
    process=Process.sequential
)

# Execute the orchestrator-worker pattern
print("=" * 60)
print("PATTERN D: ORCHESTRATOR-WORKER EXAMPLE")
print("=" * 60)
print("Manager breaks down task → Worker executes each step")
print("=" * 60 + "\n")

result = await orchestrator_crew.kickoff_async(inputs={"topic": "The Future of Renewable Energy"})

print("\n" + "=" * 60)
print("FINAL BLOG POST:")
print("=" * 60)
print(result)

PATTERN D: ORCHESTRATOR-WORKER EXAMPLE
Manager breaks down task → Worker executes each step



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.28                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e81cdc24-4aed-47a8-a5f5-5e0a2a5bbf8e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a plan for writing a blog post about The Future of Renewable Energy. List the key steps needed    │
│  (research, outline, draft, edit).                                                                              │
│  ID: 27e55d08-8b47-4fb6-b9f0-45b89838cc68                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Task: Create a plan for writing a blog post about The Future of Renewable Energy. List the key steps needed    │
│  (research, outline, draft, edit).                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (20 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Worker                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The Future of Renewable Energy                                                                               │
│                                                                                                                 │
│  Renewable energy—power from sources like solar, wind, water, and geothermal—is no longer a niche alternative.  │
│  It is becoming a mainstream part of the global energy mix, and its role will only continue to grow in the      │
│  years ahead. The future of energy will be shaped by a few critical factors: affordability, scalability,        │
│  storage, and the infrastructure needed to deliver power reliably. So what will drive renewable energy growth   │
│  next?                                                                                                          │
│                                                                                                                 │
│  The short answer is a mix of economics, policy, and technology.                                                │
│                                                                                                                 │
│  Renewables are already becoming the dominant source of new power capacity around the world. Solar PV and wind  │
│  are leading the charge, with record-setting additions in recent years. In many markets, they are not just the  │
│  cleanest option—they are also the cheapest and fastest to deploy. That matters because the energy transition   │
│  is now being driven by both climate goals and market forces.                                                   │
│                                                                                                                 │
│  Solar is likely to lead the way. Costs have fallen dramatically since 2010, and solar can be deployed at       │
│  nearly any scale, from rooftop systems to massive utility projects. Add solar-plus-storage, and homes and      │
│  businesses gain greater resilience and energy security.                                                        │
│                                                                                                                 │
│  Wind will remain essential too. Onshore wind is among the lowest-cost sources of new electricity, while        │
│  offshore wind offers huge potential near coastal demand centers. New turbine designs, better monitoring, and   │
│  improved maintenance are making wind even more competitive.                                                    │
│                                                                                                                 │
│  But renewables need storage. Batteries and other technologies will be crucial for balancing supply and demand  │
│  when the sun isn’t shining or the wind isn’t blowing. Long-duration storage, green hydrogen, and smart grids   │
│  will all play a growing role in making renewable power more reliable and flexible.                             │
│                                                                                                                 │
│  The future of renewable energy is not just about generating clean power—it’s about building a smarter, more    │
│  resilient energy system. And that future is already taking shape.                                              │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: SUBTASK 4: Review and edit the draft. Improve clarity, fix any issues, and ensure the blog post is       │
│  polished and ready for publication.                                                                            │
│  Agent: Content Worker                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL BLOG POST:
# The Future of Renewable Energy

Renewable energy—power from sources like solar, wind, water, and geothermal—is no longer a niche alternative. It is becoming a mainstream part of the global energy mix, and its role will only continue to grow in the years ahead. The future of energy will be shaped by a few critical factors: affordability, scalability, storage, and the infrastructure needed to deliver power reliably. So what will drive renewable energy growth next?

The short answer is a mix of economics, policy, and technology.

Renewables are already becoming the dominant source of new power capacity around the world. Solar PV and wind are leading the charge, with record-setting additions in recent years. In many markets, they are not just the cleanest option—they are also the cheapest and fastest to deploy. That matters because the energy transition is now being driven by both climate goals and market forces.

Solar is likely to lead the way. Costs have fallen drama

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e81cdc24-4aed-47a8-a5f5-5e0a2a5bbf8e                                                                       │
│  Final Output: # The Future of Renewable Energy                                                                 │
│                                                                                                                 │
│  Renewable energy—power from sources like solar, wind, water, and geothermal—is no longer a niche alternative.  │
│  It is becoming a mainstream part of the global energy mix, and its role will only continue to grow in the      │
│  years ahead. The future of energy will be shaped by a few critical factors: affordability, scalability,        │
│  storage, and the infrastructure needed to deliver power reliably. So what will drive renewable energy growth   │
│  next?                                                                                                          │
│                                                                                                                 │
│  The short answer is a mix of economics, policy, and technology.                                                │
│                                                                                                                 │
│  Renewables are already becoming the dominant source of new power capacity around the world. Solar PV and wind  │
│  are leading the charge, with record-setting additions in recent years. In many markets, they are not just the  │
│  cleanest option—they are also the cheapest and fastest to deploy. That matters because the energy transition   │
│  is now being driven by both climate goals and market forces.                                                   │
│                                                                                                                 │
│  Solar is likely to lead the way. Costs have fallen dramatically since 2010, and solar can be deployed at       │
│  nearly any scale, from rooftop systems to massive utility projects. Add solar-plus-storage, and homes and      │
│  businesses gain greater resilience and energy security.                                                        │
│                                                                                                                 │
│  Wind will remain essential too. Onshore wind is among the lowest-cost sources of new electricity, while        │
│  offshore wind offers huge potential near coastal demand centers. New turbine designs, better monitoring, and   │
│  improved maintenance are making wind even more competitive.                                                    │
│                                                                                                                 │
│  But renewables need storage. Batteries and other technologies will be crucial for balancing supply and demand  │
│  when the sun isn’t shining or the wind isn’t blowing. Long-duration storage, green hydrogen, and smart grids   │
│  will all play a growing role in making renewable power more reliable and flexible.                             │
│                                                                                                                 │
│  The future of renewable energy is not just about generating clean power—it’s about building a smarter, more    │
│  resilient energy system. And that future is already taking shape.                                              │
│                                                       

## 7. Pattern E: Evaluator-Optimizer

**Concept**: An iterative feedback loop where one agent generates content, another evaluates it, and the generator improves based on feedback.

**Use Case**: When output quality is critical and needs iterative refinement.

**Example**: Content Generator → Evaluator → Improved Generator (Feedback Loop)

In [8]:
# Generator Agent: Creates initial content
generator_agent = Agent(
    role="Content Generator",
    goal="Create compelling marketing copy for {product}",
    backstory="You are a creative copywriter who generates engaging marketing content. "
              "You're open to feedback and continuously improve your work.",
    llm=llm,
    verbose=True
)

# Evaluator Agent: Critiques the content
evaluator_agent = Agent(
    role="Marketing Copy Evaluator",
    goal="Evaluate marketing copy and provide constructive feedback",
    backstory="You are a seasoned marketing director with a keen eye for effective copy. "
              "You evaluate content based on: clarity, persuasiveness, emotional appeal, "
              "and call-to-action strength.",
    llm=llm,
    verbose=True
)

# Optimizer Agent: Improves based on feedback
optimizer_agent = Agent(
    role="Content Optimizer",
    goal="Improve marketing copy based on evaluator feedback",
    backstory="You are a skilled editor who excels at refining content based on feedback. "
              "You make targeted improvements while maintaining the original creative vision.",
    llm=llm,
    verbose=True
)

# Iteration 1: Generate initial content
generation_task = Task(
    description="Write a compelling 100-word marketing copy for {product}. "
                "Include a strong headline and a clear call-to-action.",
    agent=generator_agent,
    expected_output="Initial marketing copy with headline and call-to-action"
)

# Iteration 2: Evaluate the content
evaluation_task = Task(
    description="Evaluate the marketing copy. Rate it on:\n"
                "1. Clarity (1-10)\n"
                "2. Persuasiveness (1-10)\n"
                "3. Emotional Appeal (1-10)\n"
                "4. Call-to-Action Strength (1-10)\n\n"
                "Provide specific suggestions for improvement.",
    agent=evaluator_agent,
    expected_output="Detailed evaluation with ratings and improvement suggestions",
    context=[generation_task]
)

# Iteration 3: Optimize based on feedback
optimization_task = Task(
    description="Improve the marketing copy based on the evaluator's feedback. "
                "Address each suggestion and create a refined version.",
    agent=optimizer_agent,
    expected_output="Improved marketing copy incorporating feedback",
    context=[generation_task, evaluation_task]
)

# Iteration 4: Re-evaluate the improved version
final_evaluation_task = Task(
    description="Re-evaluate the improved marketing copy. Compare it to the original "
                "and confirm the improvements. Provide final ratings.",
    agent=evaluator_agent,
    expected_output="Final evaluation confirming improvements",
    context=[optimization_task]
)

# Create the evaluator-optimizer crew
feedback_loop_crew = Crew(
    agents=[generator_agent, evaluator_agent, optimizer_agent],
    tasks=[generation_task, evaluation_task, optimization_task, final_evaluation_task],
    verbose=True,
    process=Process.sequential
)

# Execute the feedback loop
print("=" * 60)
print("PATTERN E: EVALUATOR-OPTIMIZER EXAMPLE")
print("=" * 60)
print("Generate → Evaluate → Optimize → Re-evaluate")
print("=" * 60 + "\n")

result = await feedback_loop_crew.kickoff_async(
    inputs={"product": "An AI-powered personal finance app that helps users save money"}
)

print("\n" + "=" * 60)
print("FINAL RESULT (After Feedback Loop):")
print("=" * 60)
print(result)

PATTERN E: EVALUATOR-OPTIMIZER EXAMPLE
Generate → Evaluate → Optimize → Re-evaluate



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.28                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c3937b2c-6ad6-46a8-8e14-f87a723b2ee1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Write a compelling 100-word marketing copy for An AI-powered personal finance app that helps users save  │
│  money. Include a strong headline and a clear call-to-action.                                                   │
│  ID: 9c24376d-6167-4e10-a819-d575fc6302f9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Generator                                                                                       │
│                                                                                                                 │
│  Task: Write a compelling 100-word marketing copy for An AI-powered personal finance app that helps users save  │
│  money. Include a strong headline and a clear call-to-action.                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


... (15 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Copy Evaluator                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The improved copy is **stronger than the original in nearly every dimension**. It communicates the value       │
│  proposition more clearly, feels more benefit-driven, and ends with a more direct, action-oriented CTA.         │
│                                                                                                                 │
│  ### What Improved                                                                                              │
│                                                                                                                 │
│  **1. Clarity**                                                                                                 │
│  - The headline, **“Take Control of Your Money—Automatically,”** is clear and compelling.                       │
│  - The copy quickly explains what the app does: tracks spending, uncovers waste, and provides personalized      │
│  recommendations.                                                                                               │
│  - The dashboard and feature descriptions are easy to scan and understand.                                      │
│                                                                                                                 │
│  **2. Persuasiveness**                                                                                          │
│  - The improved version does a better job connecting features to outcomes:                                      │
│    - save more with less effort                                                                                 │
│    - make better money moves every day                                                                          │
│    - feel confident about every dollar                                                                          │
│  - This benefit-led framing makes the app feel more useful and relevant.                                        │
│                                                                                                                 │
│  **3. Emotional Appeal**                                                                                        │
│  - It taps into common financial emotions: stress, uncertainty, and the desire for control.                     │
│  - Phrases like **“feel confident about every dollar”** and **“simplifying their finances”** create             │
│  reassurance and motivation.                                                                                    │
│                                                                                                                 │
│  **4. Call to Action**                                                                                          │
│  - The CTA is much stronger:                                                                                    │
│    - **“Join thousands of users…”** adds social proof                                                           │
│    - **“Start saving smarter today—download the app now.”** is clear, urgent, and specific                      │
│  - This is a meaningful improvement over a weaker or more generic closing.                                      │
│                                                                                                                 │
│  ---                                                   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Re-evaluate the improved marketing copy. Compare it to the original and confirm the improvements.        │
│  Provide final ratings.                                                                                         │
│  Agent: Marketing Copy Evaluator                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL RESULT (After Feedback Loop):
The improved copy is **stronger than the original in nearly every dimension**. It communicates the value proposition more clearly, feels more benefit-driven, and ends with a more direct, action-oriented CTA.

### What Improved

**1. Clarity**
- The headline, **“Take Control of Your Money—Automatically,”** is clear and compelling.
- The copy quickly explains what the app does: tracks spending, uncovers waste, and provides personalized recommendations.
- The dashboard and feature descriptions are easy to scan and understand.

**2. Persuasiveness**
- The improved version does a better job connecting features to outcomes:
  - save more with less effort
  - make better money moves every day
  - feel confident about every dollar
- This benefit-led framing makes the app feel more useful and relevant.

**3. Emotional Appeal**
- It taps into common financial emotions: stress, uncertainty, and the desire for control.
- Phrases like **“feel confident about eve

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c3937b2c-6ad6-46a8-8e14-f87a723b2ee1                                                                       │
│  Final Output: The improved copy is **stronger than the original in nearly every dimension**. It communicates   │
│  the value proposition more clearly, feels more benefit-driven, and ends with a more direct, action-oriented    │
│  CTA.                                                                                                           │
│                                                                                                                 │
│  ### What Improved                                                                                              │
│                                                                                                                 │
│  **1. Clarity**                                                                                                 │
│  - The headline, **“Take Control of Your Money—Automatically,”** is clear and compelling.                       │
│  - The copy quickly explains what the app does: tracks spending, uncovers waste, and provides personalized      │
│  recommendations.                                                                                               │
│  - The dashboard and feature descriptions are easy to scan and understand.                                      │
│                                                                                                                 │
│  **2. Persuasiveness**                                                                                          │
│  - The improved version does a better job connecting features to outcomes:                                      │
│    - save more with less effort                                                                                 │
│    - make better money moves every day                                                                          │
│    - feel confident about every dollar                                                                          │
│  - This benefit-led framing makes the app feel more useful and relevant.                                        │
│                                                                                                                 │
│  **3. Emotional Appeal**                                                                                        │
│  - It taps into common financial emotions: stress, uncertainty, and the desire for control.                     │
│  - Phrases like **“feel confident about every dollar”** and **“simplifying their finances”** create             │
│  reassurance and motivation.                                                                                    │
│                                                                                                                 │
│  **4. Call to Action**                                                                                          │
│  - The CTA is much stronger:                                                                                    │
│    - **“Join thousands of users…”** adds social proof                                                           │
│    - **“Start saving smarter today—download the app now.”** is clear, urgent, and specific                      │
│  - This is a meaningful improvement over a weaker or more generic closing.                                      │
│                                                       

## 8. Conclusion: When to Use Each Pattern

Now that we've explored the five fundamental agentic patterns, let's summarize when to use each one:

### Pattern Selection Guide

| Pattern | Best Used When | Key Benefits | Example Use Cases |
|---------|---------------|--------------|-------------------|
| **Prompt Chaining** | Tasks naturally flow in sequence, each step depends on the previous | Simple, predictable, easy to debug | Data pipeline (extract → transform → load), Research → Summary |
| **Routing** | You have multiple specialized agents and need dynamic selection | Efficient resource use, specialized expertise | Customer service routing, Intent classification, Dynamic tool selection |
| **Parallelization** | Independent subtasks can run simultaneously | Time efficiency, concurrent processing | Market research + competitor analysis, Multiple data source aggregation |
| **Orchestrator-Worker** | Complex tasks need decomposition into managed steps | Clear structure, scalable delegation | Project planning, Multi-step content creation, Complex workflows |
| **Evaluator-Optimizer** | Quality is critical and iterative improvement needed | High-quality output, continuous refinement | Content creation, Code review, Strategic planning |

### Combining Patterns

These patterns aren't mutually exclusive! In production systems, you might combine them:
- **Routing + Parallelization**: Route queries to multiple specialists who work in parallel
- **Orchestrator-Worker + Evaluator-Optimizer**: Orchestrate a process with quality checks at each step
- **Prompt Chaining + Routing**: Chain multiple steps where each step routes to different specialists

### Key Takeaways

1. **Start Simple**: Begin with Prompt Chaining for linear workflows
2. **Add Intelligence**: Use Routing when you need dynamic decision-making
3. **Scale Performance**: Apply Parallelization for time-critical operations
4. **Manage Complexity**: Use Orchestrator-Worker for complex, multi-step tasks
5. **Ensure Quality**: Implement Evaluator-Optimizer when output quality is paramount

### Next Steps

- Experiment with each pattern using your own use cases
- Try combining patterns for more sophisticated workflows
- Explore CrewAI's additional features: custom tools, memory, and advanced configurations
- Monitor agent performance and iterate on your designs

Happy building with CrewAI! 🚀